# RAG Pipeline From Scratch
**Topics covered:** what RAG is and why · the 2 phases (index, query) · retrieval · building the prompt · calling the LLM · returning sources

> Last notebooks: you built embeddings, chunking and a vector database. Now we plug an LLM on top. That is RAG.

---
## PART 1: What Is RAG?

**RAG = Retrieval-Augmented Generation.**

An LLM only knows what it was trained on. It has never seen your company handbook, your notes, or yesterday's news. If you ask it, it will either say it doesn't know, or worse, **make something up**.

**Analogy: open-book exam.**
- Plain LLM = a student writing from memory.
- RAG = the same student, but before answering they **look up the right pages** in the book and answer from those.

The three steps in the name:
1. **Retrieve:** find the chunks that are relevant to the question
2. **Augment:** paste those chunks into the prompt
3. **Generate:** the LLM writes the answer using them

### The whole pipeline

```
INDEX (done once, ahead of time)
  documents -> split into chunks -> embed each chunk -> store in vector DB

QUERY (done on every question)
  question -> embed -> search vector DB -> top-k chunks -> build prompt -> LLM -> answer + sources
```

You already built the INDEX side and the search. The only new pieces are the **prompt** and the **LLM call**.

**What these two words mean here:**
- **INDEX** = the whole offline prep pipeline (split, embed, store). It *writes* to the vector DB, once.
- **QUERY** = the whole live pipeline that answers a user's question (embed the question, search, prompt, LLM). It *reads* from the vector DB, on every question.


---
## PART 2: Our Knowledge Base

We use a made-up company handbook, "Nimbus Labs". It is made up on purpose: no LLM has ever seen it, so if an answer is correct, it came from retrieval and not from memory.

In [17]:
handbook = """
## Annual Leave
Every full-time employee gets 24 days of paid annual leave per year. Up to 5 unused days can be carried over to the next year; the rest expire on 31 March. Leave of more than 3 days must be requested in the Pulse portal at least 2 weeks in advance.

## Sick Leave
Employees get 10 days of paid sick leave per year. A doctor's note is needed if you are out for more than 2 days in a row. Sick days do not carry over.

## Parental Leave
The primary caregiver gets 26 weeks of fully paid parental leave. The secondary caregiver gets 4 weeks. Leave can be taken any time in the first year after the child arrives.

## Public Holidays
Nimbus Labs observes 12 public holidays a year. The full list is published on the Pulse portal every January. If a holiday falls on a weekend, no extra day off is given.

## Remote Work
Teams may work remotely up to 3 days a week. Everyone must be reachable during core hours, 11:00 to 15:00 IST. Tuesday is the office day for all teams, so plan in-person meetings for that day.

## Expenses
Submit expense claims within 30 days of spending. Receipts are required for any item over Rs 2,000. While travelling for work, meals are reimbursed up to Rs 1,200 per day. Claims are filed in the Pulse portal.

## Travel
Book flights at least 14 days ahead through the travel desk. Economy class only for flights under 6 hours. Hotels are capped at Rs 7,000 per night in metro cities.

## Equipment
New joiners get a laptop on day one. Laptops are refreshed every 3 years. A one-time home office stipend of Rs 15,000 is available to set up a desk and chair.

## Learning Budget
Each employee has a learning budget of Rs 40,000 per year for courses, books and conferences. Purchases above Rs 10,000 need manager approval first.

## Device Security
If your laptop or phone is lost or stolen, report it to the security team within 24 hours. The team will remotely lock and wipe the device. Never leave a laptop unattended in a car.

## Network and Passwords
Staff connect to the NimbusSecure WiFi network; visitors use NimbusGuest. A VPN is required to reach production systems. Passwords must be changed every 90 days.

## On-Call
Engineers rotate on-call duty weekly. The on-call allowance is Rs 5,000 per week. For a P1 critical incident, the on-call engineer must respond within 15 minutes.

## Performance Reviews
Performance reviews happen twice a year, in April and October. Each review gives a rating from 1 to 5 and a written summary from your manager.

## Onboarding
Every new joiner is assigned a buddy for the first week and follows a 30-60-90 day plan agreed with their manager.

## Health Insurance
Health insurance covers the employee, spouse and up to 2 children with a Rs 5,00,000 cover. Enrolment must be completed within 30 days of joining.

## Referral Bonus
Employees who refer a successful hire receive a Rs 25,000 bonus after the new hire completes 6 months.
"""

print(len(handbook), "characters,", handbook.count("##"), "sections")

2918 characters, 16 sections


---
## PART 3: Index Phase (Chunk, Embed, Store)

Each handbook section is short and self-contained, so we use one section as one chunk. Splitting on blank lines is enough here.

In [18]:
chunks = [c.strip() for c in handbook.split("\n\n") if c.strip()]
print(len(chunks), "chunks")
print()
print(chunks[0])

16 chunks

## Annual Leave
Every full-time employee gets 24 days of paid annual leave per year. Up to 5 unused days can be carried over to the next year; the rest expire on 31 March. Leave of more than 3 days must be requested in the Pulse portal at least 2 weeks in advance.


In [19]:
from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()                                   # in-memory
collection = client.get_or_create_collection("handbook", metadata={"hnsw:space": "cosine"})

collection.add(
    ids=[f"chunk{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=model.encode(chunks, normalize_embeddings=True).tolist(),
)
print("stored:", collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

stored: 16


---
## PART 4: Retrieval

Embed the question with the **same model**, then ask the vector DB for the closest chunks.

In [20]:
def retrieve(question, k=3):
    q_vec = model.encode(question, normalize_embeddings=True).tolist()
    result = collection.query(query_embeddings=[q_vec], n_results=k)
    docs = result["documents"][0]
    sims = [1 - d for d in result["distances"][0]]            # distance -> similarity
    return list(zip(docs, sims))

for text, sim in retrieve("How many vacation days do I get?"):
    print(f"{sim:.3f}  {text[:90]}...")

0.442  ## Sick Leave
Employees get 10 days of paid sick leave per year. A doctor's note is needed...
0.433  ## Annual Leave
Every full-time employee gets 24 days of paid annual leave per year. Up to...
0.420  ## Public Holidays
Nimbus Labs observes 12 public holidays a year. The full list is publis...


The question says "vacation days" but the handbook says "annual leave". Embeddings match on **meaning**, so it still finds the right section.

---
## PART 5: Build the Prompt (the "Augment" Step)

The prompt has three parts:

1. **Instructions:** answer only from the context, say "I don't know" if it isn't there, cite the numbers
2. **Context:** the retrieved chunks, numbered so the model can cite them
3. **Question**

The "only from the context" instruction is what keeps the LLM from making things up.

In [21]:
SYSTEM = (
    "You answer questions about the Nimbus Labs employee handbook. "
    "Use ONLY the context provided. "
    "If the answer is not in the context, reply exactly: I don't know based on the handbook. "
    "Cite the context numbers you used, like [1]."
)

def build_prompt(question, retrieved):
    context = "\n\n".join(f"[{i + 1}] {text}" for i, (text, _) in enumerate(retrieved))
    return f"Context:\n{context}\n\nQuestion: {question}"

question = "How many vacation days do I get?"
user_prompt = build_prompt(question, retrieve(question, k=2))
print(user_prompt)

Context:
[1] ## Sick Leave
Employees get 10 days of paid sick leave per year. A doctor's note is needed if you are out for more than 2 days in a row. Sick days do not carry over.

[2] ## Annual Leave
Every full-time employee gets 24 days of paid annual leave per year. Up to 5 unused days can be carried over to the next year; the rest expire on 31 March. Leave of more than 3 days must be requested in the Pulse portal at least 2 weeks in advance.

Question: How many vacation days do I get?


---
## PART 6: Call the LLM (the "Generate" Step)

We use **Groq**, which runs open models (like Llama) fast and has a free tier. Setup, which you do yourself:

1. Create a free key at console.groq.com
2. Install the library: `pip install groq`
3. Put the key in the `.env` file at the project root: `GROQ_API_KEY=...`

If no key is found, the notebook still runs: it skips the LLM call and tells you so, so every other cell keeps working.

> **Model names change.** `MODEL` is set in one place below. If you get a "model not found" error, pick a current chat model from the Groq console and paste its name there.

In [22]:
import os
from dotenv import load_dotenv

load_dotenv()                                                 # reads GROQ_API_KEY from .env
HAS_KEY = bool(os.getenv("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-120b"                            # check console.groq.com for the current model list

if HAS_KEY:
    from groq import Groq
    llm = Groq()                                              # picks the key up from the environment

print("LLM available:", HAS_KEY)

LLM available: True


In [23]:
def ask_llm(system, user):
    if not HAS_KEY:
        return "[dry run: no GROQ_API_KEY found, so no LLM was called]"
    response = llm.chat.completions.create(
        model=MODEL,
        max_tokens=1000,
        temperature=0,                                        # same question -> same answer, easier to test
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
    )
    return response.choices[0].message.content

print(ask_llm(SYSTEM, user_prompt))

You receive 24 days of paid annual (vacation) leave each year. 【2】


---
## PART 7: The Full Pipeline in One Function

Retrieve, build the prompt, generate, and **return the sources** so the user can check the answer.

We also add a guard: if even the best chunk is a weak match, the question is probably not covered by the handbook, so we skip the LLM and say so. This saves money and avoids guessing.

In [24]:
def rag_answer(question, k=3, min_similarity=0.25):
    retrieved = retrieve(question, k)
    best = retrieved[0][1]
    if best < min_similarity:                                 # nothing relevant found
        return {"answer": "I don't know based on the handbook.", "sources": [], "best_similarity": best}
    answer = ask_llm(SYSTEM, build_prompt(question, retrieved))
    return {"answer": answer, "sources": [t.splitlines()[0] for t, _ in retrieved], "best_similarity": best}

for q in ["How many vacation days do I get?",
          "What's the daily meal limit when I travel?",
          "What is the capital of France?"]:
    out = rag_answer(q)
    print("Q:", q)
    print("A:", out["answer"])
    print("sources:", out["sources"], f"(best similarity {out['best_similarity']:.2f})")
    print()

Q: How many vacation days do I get?
A: You receive 24 days of paid annual leave each year. [2]
sources: ['## Sick Leave', '## Annual Leave', '## Public Holidays'] (best similarity 0.44)

Q: What's the daily meal limit when I travel?
A: The daily meal reimbursement limit while traveling for work is **Rs 1,200 per day**. 【2】
sources: ['## Travel', '## Expenses', '## Sick Leave'] (best similarity 0.42)

Q: What is the capital of France?
A: I don't know based on the handbook.
sources: [] (best similarity 0.06)



The third question has nothing to do with the handbook, so its best similarity is low and the guard stops it. The threshold (0.25) is a starting guess. Look at the scores for in-scope and out-of-scope questions on your own data and pick a value between them.

### Why the sources matter
Showing which sections were used lets the user verify the answer. RAG answers should be checkable.

---
## PART 8: Does RAG Actually Help? (needs the API key)

Ask the same question with and without the handbook.

In [25]:
q = "How many days of paid annual leave do Nimbus Labs employees get, and how many can be carried over?"

if HAS_KEY:
    print("WITHOUT RAG:")
    print(ask_llm("Answer briefly.", q))
    print()
    print("WITH RAG:")
    print(rag_answer(q)["answer"])
else:
    print("Add GROQ_API_KEY to your .env file to run this comparison.")

WITHOUT RAG:
I’m sorry, but I can’t provide that information.

WITH RAG:
Nimbus Labs employees receive **24 days of paid annual leave per year**, and they may **carry over up to 5 unused days** to the next year [1].


Without RAG the model can't know Nimbus Labs' numbers, so it should say it doesn't know (or guess). With RAG it answers with the real figures: 24 days, 5 carried over.

---
## PART 9: When RAG Goes Wrong

| Symptom | Likely cause | Fix |
|---|---|---|
| "I don't know" but the answer IS in the docs | Retrieval missed it (bad chunking, weak embedding, k too small) | Check `retrieve()` output first, try bigger k(number of chunks), better chunks |
| Confident but wrong answer | Wrong chunk retrieved, or the model ignored the instructions | Print the prompt, strengthen the "only from context" rule |
| Answer mixes two topics | Chunks too big or k too large | Smaller chunks, lower k |
| Made-up details | Prompt allows outside knowledge | "Use ONLY the context" + "say I don't know" |
| Slow or expensive | Too many or too long chunks sent | Lower k, smaller chunks, guard threshold |

**Debugging rule:** when an answer is bad, look at the **retrieved chunks first**. If the right chunk wasn't retrieved, the LLM never had a chance. Most RAG problems are retrieval problems, not LLM problems.

**Safety note:** retrieved text is *data*, not instructions. If a document says "ignore your previous instructions", a careless prompt might obey it. Keep the instructions in the system prompt and label the retrieved text clearly as context.

---
## Key Takeaways

| Concept | Remember |
|---|---|
| **RAG** | Look up relevant text first, then let the LLM answer from it |
| **Index phase** | Chunk, embed, store. Done once |
| **Query phase** | Embed the question, retrieve top-k, build the prompt, generate |
| **Prompt** | Instructions + numbered context + question |
| **Grounding** | "Use only the context, otherwise say you don't know" |
| **Sources** | Always return them so answers can be checked |
| **Guard** | Low similarity means skip the LLM |
| **Debugging** | Check retrieval first |